# Silver Common
공용 설정 / 유틸리티. 각 Silver 노트북 첫 셀에서 `%run 00_silver_common` 으로 불러온다.
Bronze 저장 원칙(원본 불변), Silver 저장 형식(Delta Table), Job 파라미터(§Part 9)를 그대로 따른다.

### Notebook Parameters
Job/Pipeline이 실행 시 override하는 파라미터. 기본값은 단독 실행(테스트) 시 사용된다.

In [ ]:
# PARAMETERS CELL (Fabric에서 이 셀을 parameters 태그로 지정)
run_mode = "DAILY"        # INITIAL | DAILY | EVENT | BACKFILL | REPLAY
source = "ghsa"           # ghsa | epss | kev
snapshot_date = None      # "YYYY-MM-DD", None이면 오늘(UTC) 사용
target_cve_ids = []       # EVENT / BACKFILL 모드에서 대상 CVE 목록
target_ghsa_ids = []      # 특정 GHSA만 재처리할 때 (REPLAY 등)
batch_id = None           # None이면 이 노트북이 새로 발급
is_backfill = False


In [ ]:
import json, uuid, hashlib
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType, DoubleType, BooleanType, LongType

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("watchback-silver").getOrCreate()


### Lakehouse 경로 / 테이블 이름
Bronze는 `Files`(원본 형식 그대로), Silver/Gold는 Delta Table(`Tables`)에 저장한다(설계서 Part 13).

**KEV는 별도 폴링이 없다**: GitHub Commit 확인 / KEV 파일 Fetch / Hash 비교는 이미 별도 Function이 수행하고, 그 결과를 `event_type`(`kev_added`/`kev_updated`/`kev_removed_confirmed`) 단위 이벤트 파일로 `Files/bronze/kev`에 쌓아준다. Fabric은 그 폴더를 읽기만 한다 — 더 이상 "배치 목록"과 "실시간 이벤트"를 구분한 별도 경로(`kev_events`)가 필요 없다.

In [ ]:
LAKEHOUSE_FILES = "Files"

BRONZE_GHSA_PATH = f"{LAKEHOUSE_FILES}/bronze/ghsa"
BRONZE_EPSS_PATH = f"{LAKEHOUSE_FILES}/bronze/epss"
BRONZE_KEV_PATH = f"{LAKEHOUSE_FILES}/bronze/kev"  # event_type별 누적 이벤트 로그, 날짜 하위폴더 없음

SILVER_SCHEMA = "silver"

TBL = {
    # 정제 결과 (grain은 03_Gold_Schema.md 문서 대응 설계서 §2 표 참고)
    "advisory_ghsa": f"{SILVER_SCHEMA}.silver_advisory_ghsa",
    "advisory_ghsa_affected": f"{SILVER_SCHEMA}.silver_advisory_ghsa_affected",
    "vuln_alias": f"{SILVER_SCHEMA}.silver_vuln_alias",
    "package": f"{SILVER_SCHEMA}.silver_package",
    "active_cve_set": f"{SILVER_SCHEMA}.active_cve_set",
    "epss": f"{SILVER_SCHEMA}.silver_epss",
    "epss_distribution": f"{SILVER_SCHEMA}.silver_epss_distribution",
    "kev": f"{SILVER_SCHEMA}.silver_kev",
    # Validation 단계 산출물 (Reject는 raw_payload 보관, 재처리 가능해야 함 - §5.1)
    "reject_ghsa": f"{SILVER_SCHEMA}.reject_ghsa",
    "reject_epss": f"{SILVER_SCHEMA}.reject_epss",
    "reject_kev": f"{SILVER_SCHEMA}.reject_kev",
    # 노트북 간 hand-off용 staging (같은 batch 안에서만 유효, 다음 노트북이 바로 소비)
    "stg_ghsa_valid": f"{SILVER_SCHEMA}.stg_ghsa_valid",
    "stg_ghsa_normalized": f"{SILVER_SCHEMA}.stg_ghsa_normalized",
    "stg_epss_valid": f"{SILVER_SCHEMA}.stg_epss_valid",
    "stg_kev_valid": f"{SILVER_SCHEMA}.stg_kev_valid",
    "stg_new_active_cve": f"{SILVER_SCHEMA}.stg_new_active_cve",
    # 운영 메타
    "meta_batch_run": f"{SILVER_SCHEMA}.meta_batch_run",
    "meta_job_run": f"{SILVER_SCHEMA}.meta_job_run",
    "meta_dq_metric": f"{SILVER_SCHEMA}.meta_dq_metric",
    "meta_watermark": f"{SILVER_SCHEMA}.meta_watermark",
}


### 시간 / batch_id 헬퍼

In [ ]:
def now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def today_str() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")

def new_batch_id(job_name: str) -> str:
    return f"{job_name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"

if not snapshot_date:
    snapshot_date = today_str()
if not batch_id:
    batch_id = new_batch_id(f"{source}_{run_mode.lower()}")

INGESTED_AT = now_iso()


### 스키마 방어 헬퍼
Bronze Raw는 소스 API가 바뀌면 필드 유무가 달라질 수 있다(예: `published_at` vs `publishedAt`).
존재하지 않는 컬럼을 `F.col()`로 직접 참조하면 AnalysisException이 나므로, 실제 존재하는 컬럼만 후보로 삼아 `coalesce`한다.

In [ ]:
def coalesce_existing(df: DataFrame, candidates: list, default=None):
    existing = [c for c in candidates if c in df.columns]
    if not existing:
        return F.lit(default)
    return F.coalesce(*[F.col(c) for c in existing])

def existing_col(df: DataFrame, name: str, default=None):
    return F.col(name) if name in df.columns else F.lit(default)

def existing_array_col(df: DataFrame, name: str):
    return F.col(name) if name in df.columns else F.array().cast("array<string>")

def nested_or_none(df: DataFrame, path: str):
    """'data.vendorProject' 같은 점 표기 경로를 스키마에 실제로 있을 때만 안전하게 따라간다.
    GHSA(cvss.score 등)와 KEV(data.* 이벤트 payload) 양쪽에서 공용으로 쓴다."""
    from pyspark.sql.types import StructType as _StructType
    parts = path.split(".")
    if parts[0] not in df.columns:
        return F.lit(None)
    col = F.col(parts[0])
    cur_type = df.schema[parts[0]].dataType
    for p in parts[1:]:
        if not isinstance(cur_type, _StructType) or p not in cur_type.fieldNames():
            return F.lit(None)
        col = col[p]
        cur_type = cur_type[p].dataType
    return col

def nested_coalesce(df: DataFrame, paths: list):
    return F.coalesce(*[nested_or_none(df, p) for p in paths])


### DQ Flags 배열 헬퍼
`F.when(...)` 결과들 중 NULL이 아닌 것만 모아 `dq_flags` 배열을 만든다. `array_remove`는 NULL 비교가 SQL에서 항상 False라 NULL을 못 지우므로 `F.filter` 고차함수를 쓴다.

In [ ]:
def flags_array(*when_exprs):
    return F.filter(F.array(*when_exprs), lambda x: x.isNotNull())


### Content Hash 헬퍼
`silver_advisory_ghsa`, `silver_advisory_ghsa_affected`, `silver_kev`의 `record_hash`에 공통으로 쓴다.
**주의**: `batch_id`/`ingested_at`/`record_hash` 자신은 절대 해시 대상에 넣지 않는다 — 넣으면 실행할 때마다 값이 달라져 항상 새 버전으로 오판된다.

In [ ]:
def content_hash(cols: list, length: int = 16):
    """cols: 해시에 포함할 컬럼(Column) 리스트. 순서가 바뀌어도 값이 같으면 같은 해시가 나오도록 정렬 가능한 JSON으로 직렬화한다."""
    return F.substring(F.sha2(F.to_json(F.struct(*cols)), 256), 1, length)


### Delta 쓰기 헬퍼

In [ ]:
def table_exists(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def write_overwrite(df: DataFrame, table_name: str, partition_by=None):
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.saveAsTable(table_name)

def write_append(df: DataFrame, table_name: str, partition_by=None):
    writer = df.write.format("delta").mode("append").option("mergeSchema", "true")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.saveAsTable(table_name)


### SCD2 Merge 헬퍼
`silver_advisory_ghsa`, `silver_kev`처럼 "값이 바뀔 때만 새 버전"을 만드는 Change History 테이블에 공통으로 쓴다(설계서 Part 11).
- 키가 매칭되고 해시가 다르면: 기존 열린 버전(`valid_to IS NULL`)을 닫고, 새 버전을 연다.
- 키가 매칭되지 않으면: 새 버전(신규 엔터티)을 연다.
- 해시가 같으면: 아무 것도 하지 않는다(불필요한 버전 분기 방지 — `silver_kev` 노이즈 문제, 03_Gold_Schema.md 참고).

In [ ]:
from delta.tables import DeltaTable

def merge_scd2(new_df: DataFrame, table_name: str, key_cols: list,
                hash_col: str = "record_hash",
                valid_from_col: str = "valid_from",
                valid_to_col: str = "valid_to") -> dict:
    if not table_exists(table_name):
        first_version = new_df.withColumn(valid_to_col, F.lit(None).cast(StringType()))
        write_overwrite(first_version, table_name)
        n = first_version.count()
        return {"opened": n, "closed": 0, "unchanged": 0}

    target = DeltaTable.forName(spark, table_name)
    current_open = target.toDF().filter(F.col(valid_to_col).isNull())

    join_cond = [current_open[k] == new_df[k] for k in key_cols]
    compared = (
        current_open.alias("cur")
        .join(new_df.alias("new"), join_cond, "inner")
        .select(
            *[F.col(f"new.{k}").alias(k) for k in key_cols],
            F.col(f"cur.{hash_col}").alias("cur_hash"),
            F.col(f"new.{hash_col}").alias("new_hash"),
            F.col(f"new.{valid_from_col}").alias("new_valid_from"),
        )
    )

    changed = compared.filter(F.col("cur_hash") != F.col("new_hash"))
    changed_count = changed.count()

    if changed_count > 0:
        changed.select(*key_cols, F.col("new_valid_from").alias("close_at")) \
            .createOrReplaceTempView("_scd2_close_updates")
        merge_keys = " AND ".join([f"t.{k} = s.{k}" for k in key_cols])
        merge_cond = f"{merge_keys} AND t.{valid_to_col} IS NULL"
        (
            target.alias("t")
            .merge(spark.table("_scd2_close_updates").alias("s"), merge_cond)
            .whenMatchedUpdate(set={valid_to_col: "s.close_at"})
            .execute()
        )

    cur_hash_lookup = current_open.select(*key_cols, F.col(hash_col).alias("cur_hash"))
    new_or_changed = (
        new_df.join(cur_hash_lookup, on=key_cols, how="left")
        .filter(F.col("cur_hash").isNull() | (F.col("cur_hash") != F.col(hash_col)))
        .drop("cur_hash")
        .withColumn(valid_to_col, F.lit(None).cast(StringType()))
    )
    opened_count = new_or_changed.count()
    if opened_count > 0:
        write_append(new_or_changed, table_name)

    return {"opened": opened_count, "closed": changed_count}


### Watermark / Checkpoint 헬퍼
KEV Bronze가 날짜 폴더 없이 계속 누적되는 Event Log이므로, 매번 폴더 전체를 다시 읽으면 안 된다.
`meta_watermark`에 소스별 마지막 처리 지점(`last_processed_file`, `last_processed_timestamp`)을 기록해두고, 다음 실행에서는 그 이후 파일만 읽는다.

In [ ]:
def get_watermark(wm_source: str):
    if not table_exists(TBL["meta_watermark"]):
        return None
    rows = spark.table(TBL["meta_watermark"]).filter(F.col("source") == wm_source).collect()
    return rows[0] if rows else None

def update_watermark(wm_source: str, last_processed_file: str, last_processed_timestamp: str):
    row = spark.createDataFrame([{
        "source": wm_source,
        "last_processed_file": last_processed_file,
        "last_processed_timestamp": last_processed_timestamp,
        "updated_at": now_iso(),
    }])
    if table_exists(TBL["meta_watermark"]):
        target = DeltaTable.forName(spark, TBL["meta_watermark"])
        (
            target.alias("t")
            .merge(row.alias("s"), "t.source = s.source")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        write_overwrite(row, TBL["meta_watermark"])


### 운영 메타 로깅
Job Dependency(§Part 8), Job 실행 상태 관리(§Part 7)에서 정의한 `meta_batch_run`에 매 노트북 실행 결과를 남긴다.
`gold_circuit_breaker_check` 및 `reconcile_silver`가 이 로그를 근거로 Publish 여부를 판단한다.

In [ ]:
def log_batch_run(job_name: str, status: str, record_count: int = 0, error_message: str = None):
    row = spark.createDataFrame([{
        "batch_id": batch_id,
        "job_name": job_name,
        "source": source,
        "run_mode": run_mode,
        "started_at": INGESTED_AT,
        "finished_at": now_iso(),
        "status": status,
        "record_count": int(record_count),
        "error_message": error_message,
    }])
    write_append(row, TBL["meta_batch_run"])

def log_dq_metric(job_name: str, metric_name: str, metric_value, passed: bool, detail: str = None):
    row = spark.createDataFrame([{
        "batch_id": batch_id,
        "job_name": job_name,
        "snapshot_date": snapshot_date,
        "metric_name": metric_name,
        "metric_value": str(metric_value),
        "passed": bool(passed),
        "detail": detail,
        "computed_at": now_iso(),
    }])
    write_append(row, TBL["meta_dq_metric"])
